In [1]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine

In [2]:

flights_data = pd.read_csv('../Flight_Data/flight_data_2024_sample.csv')
weather_data = pd.read_csv('../Weather_Data/weather_data_sample.csv')


In [3]:
weather_data.head()

,Location,Date_Time,Temperature_C,Humidity_pct,Precipitation_mm,Wind_Speed_kmh
0,Chicago,2024-03-10 17:21:58,16.296300,78.971811,7.620581,15.928304
1,Los Angeles,2024-02-07 10:27:25,11.114736,87.441769,2.900566,3.381471
2,Phoenix,2024-04-27 06:26:35,25.639647,51.484221,3.615657,28.163489
3,Los Angeles,2024-03-30 05:43:57,30.696139,41.867403,2.813543,16.502607
4,San Antonio,2024-04-19 03:03:02,-5.572413,52.167586,0.256806,7.343845


In [54]:
weather_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Location          30000 non-null  object 
 1   Date_Time         30000 non-null  object 
 2   Temperature_C     30000 non-null  float64
 3   Humidity_pct      30000 non-null  float64
 4   Precipitation_mm  30000 non-null  float64
 5   Wind_Speed_kmh    30000 non-null  float64
dtypes: float64(4), object(2)
memory usage: 1.4+ MB


In [6]:

print(flights_data.columns)

Index(['year', 'month', 'day_of_month', 'day_of_week', 'fl_date',
       'op_unique_carrier', 'op_carrier_fl_num', 'origin', 'origin_city_name',
       'origin_state_nm', 'dest', 'dest_city_name', 'dest_state_nm',
       'crs_dep_time', 'dep_time', 'dep_delay', 'taxi_out', 'wheels_off',
       'wheels_on', 'taxi_in', 'crs_arr_time', 'arr_time', 'arr_delay',
       'cancelled', 'cancellation_code', 'diverted', 'crs_elapsed_time',
       'actual_elapsed_time', 'air_time', 'distance', 'carrier_delay',
       'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay'],
      dtype='object')


In [8]:
print(flights_data)

      year  month  day_of_month  day_of_week     fl_date op_unique_carrier  \
0     2024      4            18            4  2024-04-18                MQ   
1     2024      1             1            1  2024-01-01                AA   
2     2024     12            12            4  2024-12-12                9E   
3     2024      4             8            1  2024-04-08                WN   
4     2024      2            16            5  2024-02-16                WN   
...    ...    ...           ...          ...         ...               ...   
9995  2024      1            16            2  2024-01-16                WN   
9996  2024      7             5            5  2024-07-05                AA   
9997  2024      2            28            3  2024-02-28                MQ   
9998  2024      2            18            7  2024-02-18                DL   
9999  2024      3             1            5  2024-03-01                G4   

      op_carrier_fl_num origin       origin_city_name origin_st

In [7]:
flights_data.head()

,year,month,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,...,diverted,crs_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2024,4,18,4,2024-04-18,MQ,3535.0,DFW,"Dallas/Fort Worth, TX",Texas,...,0,151.0,144.0,119.0,835.0,0,0,0,0,0
1,2024,1,1,1,2024-01-01,AA,148.0,CLT,"Charlotte, NC",North Carolina,...,0,286.0,273.0,253.0,1773.0,0,0,0,0,0
2,2024,12,12,4,2024-12-12,9E,5440.0,CHA,"Chattanooga, TN",Tennessee,...,0,59.0,50.0,29.0,106.0,0,0,0,0,0
3,2024,4,8,1,2024-04-08,WN,1971.0,OMA,"Omaha, NE",Nebraska,...,0,180.0,177.0,163.0,1099.0,0,0,0,0,0
4,2024,2,16,5,2024-02-16,WN,862.0,BWI,"Baltimore, MD",Maryland,...,0,90.0,96.0,76.0,399.0,0,0,0,0,0


In [56]:
engine = create_engine('sqlite:///flights_weather.db')

In [57]:
weather_data.to_sql('weather', engine, if_exists='replace', index=False)
flights_data.to_sql('flights', engine, if_exists='replace', index=False)

10000

In [58]:
read_query = "SELECT * FROM weather LIMIT 10;"

In [ ]:
query = """
SELECT 
    Location,
    Date_Time,
    Temperature_C,
    Humidity_pct,
    Precipitation_mm,
    Wind_Speed_kmh,
    
    -- Hour (0-23)
    strftime('%H', Date_Time) AS Hour,
    
    -- Month number (01-12)
    strftime('%m', Date_Time) AS Month_Number,
    
    -- Month name (January, February, …)
    CASE strftime('%m', Date_Time)
        WHEN '01' THEN 'January'
        WHEN '02' THEN 'February'
        WHEN '03' THEN 'March'
        WHEN '04' THEN 'April'
        WHEN '05' THEN 'May'
        WHEN '06' THEN 'June'
        WHEN '07' THEN 'July'
        WHEN '08' THEN 'August'
        WHEN '09' THEN 'September'
        WHEN '10' THEN 'October'
        WHEN '11' THEN 'November'
        WHEN '12' THEN 'December'
    END AS Month_Name,
    
    -- Day of month (1-31)
    strftime('%d', Date_Time) AS Day_of_Month,
    
    -- Weekday name
    CASE strftime('%w', Date_Time)
        WHEN '0' THEN 'Sunday'
        WHEN '1' THEN 'Monday'
        WHEN '2' THEN 'Tuesday'
        WHEN '3' THEN 'Wednesday'
        WHEN '4' THEN 'Thursday'
        WHEN '5' THEN 'Friday'
        WHEN '6' THEN 'Saturday'
    END AS Weekday
FROM weather;
"""

weather_with_time = pd.read_sql(query, engine)
weather_with_time.to_sql('weather', engine, if_exists='replace', index=False)
head_weather = pd.read_sql(read_query, engine)
print(head_weather)

      Location            Date_Time  Temperature_C  Humidity_pct  \
0      Chicago  2024-03-10 17:21:58      16.296300     78.971811   
1  Los Angeles  2024-02-07 10:27:25      11.114736     87.441769   
2      Phoenix  2024-04-27 06:26:35      25.639647     51.484221   
3  Los Angeles  2024-03-30 05:43:57      30.696139     41.867403   
4  San Antonio  2024-04-19 03:03:02      -5.572413     52.167586   
5  Los Angeles  2024-03-07 16:33:01      37.925081     86.857693   
6     San Jose  2024-01-14 02:56:42      21.336537     87.068830   
7     San Jose  2024-02-28 12:40:46      -2.780308     59.550017   
8     San Jose  2024-01-13 07:04:02      11.245631     30.271421   
9      Phoenix  2024-04-07 03:17:00      28.224833     62.958570   

   Precipitation_mm  Wind_Speed_kmh Hour Month_Number Month_Name Day_of_Month  \
0          7.620581       15.928304   17           03      March           10   
1          2.900566        3.381471   10           02   February           07   
2       

In [60]:
query = """
SELECT 
    Location,
    Date_Time,
    Temperature_C,
    Humidity_pct,
    Precipitation_mm,
    Wind_Speed_kmh,

FROM flights

strftime('%H', Date_Time) AS Hour
strftime('%m', Date_Time) AS Month_Number

CASE strftime('%m', Date_Time)
    WHEN '01' THEN 'January'
    WHEN '02' THEN 'Febuary'
END AS Month_Name



"""